# Paper-style Experiment Statistics & Plots (Traffic Signal Control)

This notebook produces **paper-like figures** from your evaluation outputs (JSON) and saves them as images you can upload to Overleaf.

## What you need before running this notebook
1. **Your MARL/RL evaluation JSON** (already generated):
   - Example: `output/eval_cologne_from_vancouver_regionaware_ep120_seeded_emissions.json`

2. **Baseline evaluation JSONs** (generate these with the provided runner):
   - Script: `evaluate_baselines_osm.py`
   - You should generate one JSON per baseline (same setup, same seed protocol, emissions enabled), e.g.:
     - `output/eval_cologne_FIXED_TIME_ep120.json`
     - `output/eval_cologne_MAX_PRESSURE_ep120.json`
     - `output/eval_cologne_SOTL_ep120.json`
     - `output/eval_cologne_ADAPTIVE_ep120.json`

## Procedure (reproducible)
### A) Run baselines (example)
Run each baseline with the **same** horizon, controlled-lights ratio, and emissions settings:

```powershell
cd "D:\Final Year Project\traffic-signal-control"
python evaluate_baselines_osm.py --dataset cologne --strategy FIXED_TIME --episodes 120 --duration 1200 --seed 42 --run-id 1 --controlled-lights-ratio 0.2 --sumo-emissions-output --out output/eval_cologne_FIXED_TIME_ep120.json
python evaluate_baselines_osm.py --dataset cologne --strategy MAX_PRESSURE --episodes 120 --duration 1200 --seed 42 --run-id 1 --controlled-lights-ratio 0.2 --sumo-emissions-output --out output/eval_cologne_MAX_PRESSURE_ep120.json
python evaluate_baselines_osm.py --dataset cologne --strategy SOTL --episodes 120 --duration 1200 --seed 42 --run-id 1 --controlled-lights-ratio 0.2 --sumo-emissions-output --out output/eval_cologne_SOTL_ep120.json
python evaluate_baselines_osm.py --dataset cologne --strategy ADAPTIVE --episodes 120 --duration 1200 --seed 42 --run-id 1 --controlled-lights-ratio 0.2 --sumo-emissions-output --out output/eval_cologne_ADAPTIVE_ep120.json
```

### B) Run this notebook to generate figures
This notebook will:
- load all `output/eval_*.json` files you select
- compute **ATT/AIP/ASIP vs episode** (paper-style proxies)
- plot fundamental metrics: **CO$_2$**, **fuel**, **speed**, **waiting time** vs episode
- create a **strategy × metric heatmap** (mean performance)
- save figures to `Report/figures/nb_*.pdf` and `Report/figures/nb_*.png`

## Metric definitions used in plots
Different papers use slightly different naming. Here we use consistent proxies derived from your logged metrics:
- **ATT**: Average Travel Time proxy = `avg_waiting_time` (seconds)
- **AIP**: Average Intersection Pressure proxy = `avg_pressure`
- **ASIP**: Average System Intersection Pressure proxy = `avg_queue_length`

(You can rename these later in the report to match the exact terminology of your chosen reference paper.)


In [5]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # safe for headless export
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

ROOT = Path(".").resolve()
OUT = ROOT / "output"
FIG_DIR = ROOT / "Report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def list_eval_jsons() -> List[Path]:
    # We keep this explicit so it picks up both baseline JSONs and MARL eval JSONs.
    return sorted(OUT.glob("eval_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)


def eval_to_df(eval_json: Dict[str, Any], label: str) -> pd.DataFrame:
    rows = []
    for ep in (eval_json.get("results") or []):
        metrics = ep.get("metrics") or {}
        rows.append({
            "label": label,
            "episode": ep.get("episode"),
            "sumo_seed": ep.get("sumo_seed"),
            **metrics,
        })
    df = pd.DataFrame(rows)
    if "episode" in df.columns:
        df["episode"] = pd.to_numeric(df["episode"], errors="coerce")
    return df.sort_values(["label", "episode"])


def infer_label(path: Path, payload: Dict[str, Any]) -> str:
    # Baseline evals produced by evaluate_baselines_osm.py include a 'strategy' field.
    if payload.get("strategy"):
        return str(payload["strategy"])
    # RL evals produced by evaluate_marl_osm.py include 'model' and often indicate MARL.
    if payload.get("model"):
        return "MARL_DQN"
    # fallback: filename stem
    return path.stem


def save_fig(stem: str) -> Tuple[Path, Path]:
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.close()
    return png, pdf


def mean_std_ci(values: List[float]) -> Tuple[float, float, float, float]:
    vals = [float(v) for v in values if v is not None and math.isfinite(float(v))]
    if not vals:
        return float("nan"), float("nan"), float("nan"), float("nan")
    n = len(vals)
    mu = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if n >= 2 else 0.0
    if n >= 2:
        sem = sd / math.sqrt(n)
        t = float(stats.t.ppf(0.975, df=n - 1))
        ci_lo = mu - t * sem
        ci_hi = mu + t * sem
    else:
        ci_lo = ci_hi = mu
    return mu, sd, ci_lo, ci_hi


print("ROOT:", ROOT)
print("Eval JSONs found:")
for p in list_eval_jsons()[:20]:
    print(" -", p.name)
print("(showing up to 20; run baselines to add more)")
print("Figures will be saved to:", FIG_DIR)


ROOT: E:\Final\traffic-signal-control-main
Eval JSONs found:
 - eval_cologne_ADAPTIVE_ep120.json
 - eval_SOTL_run2.json
 - eval_SOTL_run1.json
 - eval_cologne_SOTL_ep120.json
 - eval_cologne_MAX_PRESSURE_ep120.json
 - eval_cologne_MAX_PRESSURE_test1.json
 - eval_vancouver.json
 - eval_cologne_smoketest.json
 - eval_los_angeles.json
 - eval_cologne_from_vancouver_regionaware_ep120_seeded_emissions.json
 - eval_cologne_from_vancouver_regionaware_ep120_seeded_emissions_summary.json
 - eval_cologne_from_vancouver_part12.json
 - eval_cologne_from_vancouver_regionaware.json
 - eval_cologne_from_vancouver_part10.json
 - eval_cologne_from_vancouver_part11.json
 - eval_cologne_from_vancouver_part08.json
 - eval_cologne_from_vancouver_part09.json
 - eval_cologne_from_vancouver_part07.json
 - eval_cologne_from_vancouver_part05.json
 - eval_cologne_from_vancouver_part06.json
(showing up to 20; run baselines to add more)
Figures will be saved to: E:\Final\traffic-signal-control-main\Report\figures


## 1) Load evaluation JSONs (MARL + baselines)

This notebook expects one evaluation JSON per strategy/model.

### File naming convention (recommended)
- Baselines: `output/eval_<dataset>_<STRATEGY>_ep<episodes>.json`
  - Example: `output/eval_cologne_FIXED_TIME_ep120.json`
- RL/MARL: any `output/eval_*.json` produced by `evaluate_marl_osm.py`
  - Example: `output/eval_cologne_from_vancouver_regionaware_ep120_seeded_emissions.json`

### Important
To make plots and statistics meaningful, all strategies must share the same:
- map (`--dataset`)
- horizon (`--duration`)
- controlled TLS budget (`--controlled-lights-ratio`)
- seed protocol (`--seed` + `--run-id`)
- emissions mode (`--sumo-emissions-output`)

If a baseline JSON is missing, the notebook will still plot whatever is available, but the **heatmap** and **significance tests** only become meaningful once you have multiple strategies.


In [9]:
# Select which eval JSON files to include in plots.
# Tip: start with MARL + FIXED_TIME, then add other baselines.

# Option A: explicit list (recommended)
SELECT_FILES = [
    "eval_cologne_from_vancouver_regionaware_ep120_seeded_emissions.json",  # YOUR MODEL
    "eval_cologne_FIXED_TIME_ep120.json",       # Baseline 1
    "eval_cologne_MAX_PRESSURE_ep120.json",     # Baseline 2  
    "eval_cologne_ADAPTIVE_ep120.json",         # Baseline 3
    "eval_cologne_SOTL_ep120.json",             # Baseline 4
]

# Option B: automatic discovery (uncomment if you prefer)
# SELECT_FILES = [p.name for p in list_eval_jsons() if "cologne" in p.name.lower()]

paths = []
for name in SELECT_FILES:
    p = OUT / name
    if p.exists():
        paths.append(p)
    else:
        print("Missing (skip):", p)

assert paths, "No evaluation JSON files found. Generate baseline eval JSONs first."

payloads = [load_json(p) for p in paths]
labels = [infer_label(p, pl) for p, pl in zip(paths, payloads)]

# Build combined episode-level dataframe
frames = [eval_to_df(pl, lab) for pl, lab in zip(payloads, labels)]
df_all = pd.concat(frames, ignore_index=True)

print("Loaded strategies:", sorted(df_all["label"].unique().tolist()))
print("Episodes per strategy:")
print(df_all.groupby("label")["episode"].count())

df_all.head()


Loaded strategies: ['ADAPTIVE', 'FIXED_TIME', 'MARL_DQN', 'MAX_PRESSURE', 'SOTL']
Episodes per strategy:
label
ADAPTIVE        120
FIXED_TIME      120
MARL_DQN        120
MAX_PRESSURE    120
SOTL            120
Name: episode, dtype: int64


,label,episode,sumo_seed,avg_waiting_time,max_waiting_time,total_waiting_time,avg_max_waiting_time_per_vehicle,vehicles_with_waiting,vehicles_total,percentage_vehicles_waited,...,co2_per_km,avg_lane_occupancy,lane_occupancy_std,max_lane_occupancy,avg_acceleration,acceleration_std,harsh_braking_events,vehicle_type_stats,avg_co2_per_vehicle,avg_fuel_per_vehicle
0,MARL_DQN,1,10043,104.300550,862.0,86105632.0,252.281116,6262,6307,99.286507,...,28160.243175,19.512486,28.252523,300.977353,-0.004039,0.473659,5498,"{'('waiting_time', 'mean')': {'bus_bus': 102.3...",NaN,NaN
1,MARL_DQN,2,10044,105.946962,807.0,88308912.0,256.412532,6245,6288,99.316158,...,28593.342370,19.717384,28.663097,283.237333,-0.004059,0.460404,5143,"{'('waiting_time', 'mean')': {'bus_bus': 105.1...",NaN,NaN
2,MARL_DQN,3,10045,103.640741,858.0,86609873.0,252.724771,6283,6322,99.383107,...,28326.771885,20.684438,27.142660,165.803373,-0.003392,0.467477,5372,"{'('waiting_time', 'mean')': {'bus_bus': 102.5...",NaN,NaN
3,MARL_DQN,4,10046,103.645067,752.0,86415318.0,252.229503,6274,6318,99.303577,...,28616.707428,18.290372,25.439652,170.336170,-0.003479,0.464245,5241,"{'('waiting_time', 'mean')': {'bus_bus': 103.3...",NaN,NaN
4,MARL_DQN,5,10047,103.009532,763.0,85958982.0,254.681890,6287,6328,99.352086,...,27995.126945,18.895885,25.260360,154.794498,-0.003580,0.476267,5531,"{'('waiting_time', 'mean')': {'bus_bus': 102.6...",NaN,NaN


In [10]:
# --- Paper-style plotting helpers (matplotlib/seaborn) ---

ATT = "avg_waiting_time"   # proxy for Average Travel Time
AIP = "avg_pressure"       # proxy for Average Intersection Pressure
ASIP = "avg_queue_length"  # proxy for Average System Intersection Pressure


def plot_att_aip_asip_vs_episode(df: pd.DataFrame):
    """ATT/AIP/ASIP vs episode, one subplot per metric, multiple strategies."""
    metrics = [
        (ATT, "ATT (avg waiting time, s)"),
        (AIP, "AIP (avg pressure proxy)"),
        (ASIP, "ASIP (avg queue length, veh)"),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    for ax, (m, title) in zip(axes, metrics):
        if m not in df.columns:
            ax.set_visible(False)
            continue
        for label, d in df.groupby("label"):
            d = d.sort_values("episode")
            y = pd.to_numeric(d[m], errors="coerce")
            ax.plot(d["episode"], y, alpha=0.25, linewidth=1)
            ax.plot(d["episode"], y.rolling(10, min_periods=1).mean(), linewidth=2, label=f"{label} (MA10)")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.25)

    axes[-1].set_xlabel("Episode")
    axes[0].legend(loc="best", fontsize=9)
    fig.suptitle("ATT / AIP / ASIP vs Episode", y=0.98)
    save_fig("nb_att_aip_asip_vs_episode")


def plot_fundamental_metrics_vs_episode(df: pd.DataFrame):
    """CO2, fuel, speed, waiting time vs episode."""
    wanted = [
        ("avg_waiting_time", "Average waiting time (s)"),
        ("avg_speed", "Average speed (m/s)"),
        ("total_co2", "Total CO$_2$ (mg)"),
        ("total_fuel", "Total fuel (ml)"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    axes = axes.flatten()

    for ax, (m, title) in zip(axes, wanted):
        if m not in df.columns:
            ax.set_visible(False)
            continue
        for label, d in df.groupby("label"):
            d = d.sort_values("episode")
            y = pd.to_numeric(d[m], errors="coerce")
            ax.plot(d["episode"], y.rolling(10, min_periods=1).mean(), linewidth=2, label=label)
        ax.set_title(title)
        ax.grid(True, alpha=0.25)

    axes[0].legend(loc="best", fontsize=9)
    for ax in axes:
        ax.set_xlabel("Episode")

    fig.suptitle("Fundamental metrics vs Episode (smoothed)", y=0.98)
    save_fig("nb_fundamental_metrics_vs_episode")


def plot_strategy_heatmap(df: pd.DataFrame):
    """Heatmap of mean metrics per strategy (paper-style summary)."""
    metrics = [
        "avg_waiting_time",
        "avg_queue_length",
        "throughput_per_hour",
        "congestion_index",
        "avg_speed",
        "total_co2",
        "total_fuel",
        "avg_pressure",
    ]
    keep = [m for m in metrics if m in df.columns]
    if not keep:
        print("No metrics available for heatmap")
        return

    summary = df.groupby("label")[keep].mean(numeric_only=True)

    # Normalize columns to [0,1] for visualization (not for reporting values)
    norm = (summary - summary.min()) / (summary.max() - summary.min()).replace(0, np.nan)

    plt.figure(figsize=(10, max(2.5, 0.6 * len(summary))))
    sns.heatmap(norm, annot=False, cmap="viridis", cbar=True)
    plt.title("Strategy heatmap (normalized mean metrics)")
    plt.ylabel("Strategy")
    plt.xlabel("Metric")
    save_fig("nb_strategy_metric_heatmap")


def significance_tests(df: pd.DataFrame, baseline: str = "FIXED_TIME") -> pd.DataFrame:
    """t-test and Mann-Whitney U vs baseline for key metrics."""
    metrics = ["avg_waiting_time", "avg_queue_length", "throughput_per_hour", "congestion_index", "total_co2", "total_fuel"]
    keep = [m for m in metrics if m in df.columns]

    if baseline not in df["label"].unique():
        print(f"Baseline '{baseline}' not loaded; skip significance tests")
        return pd.DataFrame()

    out_rows = []
    base_df = df[df["label"] == baseline]

    for label in sorted(df["label"].unique()):
        if label == baseline:
            continue
        d = df[df["label"] == label]
        for m in keep:
            a = pd.to_numeric(d[m], errors="coerce").dropna().values
            b = pd.to_numeric(base_df[m], errors="coerce").dropna().values
            if len(a) < 2 or len(b) < 2:
                continue
            t_stat, t_p = stats.ttest_ind(a, b, equal_var=False)
            try:
                u_stat, u_p = stats.mannwhitneyu(a, b, alternative="two-sided")
            except Exception:
                u_p = np.nan
            out_rows.append({
                "metric": m,
                "compare": f"{label} vs {baseline}",
                "mean_model": float(np.mean(a)),
                "mean_baseline": float(np.mean(b)),
                "delta": float(np.mean(a) - np.mean(b)),
                "t_p": float(t_p),
                "mw_p": float(u_p),
            })

    return pd.DataFrame(out_rows).sort_values(["metric", "compare"])


In [11]:
# Generate paper-style plots and save to Report/figures

plot_att_aip_asip_vs_episode(df_all)
plot_fundamental_metrics_vs_episode(df_all)
plot_strategy_heatmap(df_all)

# Print a simple statistics table (mean ± std, 95% CI) for key metrics
KEY = ["avg_waiting_time", "avg_queue_length", "throughput_per_hour", "congestion_index", "total_co2", "total_fuel"]
rows = []
for label, d in df_all.groupby("label"):
    for m in KEY:
        if m not in d.columns:
            continue
        mu, sd, lo, hi = mean_std_ci(d[m].tolist())
        rows.append({"label": label, "metric": m, "mean": mu, "std": sd, "ci95_low": lo, "ci95_high": hi})

stats_df = pd.DataFrame(rows)
print("\nSummary (mean±std, 95% CI):")
display(stats_df)

# Significance tests vs FIXED_TIME (requires FIXED_TIME baseline to be loaded)
print("\nSignificance tests vs FIXED_TIME (Welch t-test + Mann–Whitney U):")
res = significance_tests(df_all, baseline="FIXED_TIME")
display(res)

print("\nSaved figures to:", FIG_DIR)



Summary (mean±std, 95% CI):


,label,metric,mean,std,ci95_low,ci95_high
0,ADAPTIVE,avg_waiting_time,1.042667e+02,1.077676e+00,1.040719e+02,1.044615e+02
1,ADAPTIVE,avg_queue_length,3.025921e+03,1.240511e+01,3.023679e+03,3.028164e+03
2,ADAPTIVE,throughput_per_hour,5.210000e+02,3.631237e+01,5.144363e+02,5.275637e+02
3,ADAPTIVE,congestion_index,5.920589e-01,3.778629e-02,5.852288e-01,5.988891e-01
4,ADAPTIVE,total_co2,6.302496e+09,3.101700e+09,5.741840e+09,6.863151e+09
5,ADAPTIVE,total_fuel,2.036798e+09,1.002404e+09,1.855606e+09,2.217990e+09
6,FIXED_TIME,avg_waiting_time,1.044309e+02,1.058938e+00,1.042395e+02,1.046223e+02
7,FIXED_TIME,avg_queue_length,3.027726e+03,1.278077e+01,3.025416e+03,3.030036e+03
8,FIXED_TIME,throughput_per_hour,5.132500e+02,3.417116e+01,5.070733e+02,5.194267e+02
9,FIXED_TIME,congestion_index,6.022796e-01,4.146088e-02,5.947852e-01,6.097740e-01



Significance tests vs FIXED_TIME (Welch t-test + Mann–Whitney U):


,metric,compare,mean_model,mean_baseline,delta,t_p,mw_p
1,avg_queue_length,ADAPTIVE vs FIXED_TIME,3.025921e+03,3.027726e+03,-1.804514e+00,2.681890e-01,2.300145e-01
7,avg_queue_length,MARL_DQN vs FIXED_TIME,3.008965e+03,3.027726e+03,-1.876087e+01,7.703546e-27,1.488008e-23
13,avg_queue_length,MAX_PRESSURE vs FIXED_TIME,3.024399e+03,3.027726e+03,-3.326528e+00,5.129910e-02,7.499542e-02
19,avg_queue_length,SOTL vs FIXED_TIME,3.028564e+03,3.027726e+03,8.380556e-01,6.044137e-01,3.676205e-01
0,avg_waiting_time,ADAPTIVE vs FIXED_TIME,1.042667e+02,1.044309e+02,-1.641780e-01,2.350919e-01,1.833580e-01
6,avg_waiting_time,MARL_DQN vs FIXED_TIME,1.037804e+02,1.044309e+02,-6.504534e-01,2.640863e-06,1.242742e-06
12,avg_waiting_time,MAX_PRESSURE vs FIXED_TIME,1.042260e+02,1.044309e+02,-2.048873e-01,1.245455e-01,1.927135e-01
18,avg_waiting_time,SOTL vs FIXED_TIME,1.044482e+02,1.044309e+02,1.733949e-02,9.014749e-01,5.967823e-01
3,congestion_index,ADAPTIVE vs FIXED_TIME,5.920589e-01,6.022796e-01,-1.022064e-02,4.709696e-02,6.049268e-02
9,congestion_index,MARL_DQN vs FIXED_TIME,7.568029e-01,6.022796e-01,1.545233e-01,8.884649e-75,7.143876e-41



Saved figures to: E:\Final\traffic-signal-control-main\Report\figures


## 2) Where are the figures and how to use them in Overleaf?

After running the cells above, the notebook saves figures to:
- `Report/figures/nb_att_aip_asip_vs_episode.(pdf|png)`
- `Report/figures/nb_fundamental_metrics_vs_episode.(pdf|png)`
- `Report/figures/nb_strategy_metric_heatmap.(pdf|png)`

### Upload to Overleaf
1. Create a folder called `figures/` in Overleaf.
2. Upload the **PDF** versions (preferred for quality) from `Report/figures/`.
3. Include them in LaTeX:

```latex
\begin{figure}[h]
  \centering
  \includegraphics[width=0.95\linewidth]{figures/nb_att_aip_asip_vs_episode.pdf}
  \caption{ATT/AIP/ASIP vs episode for MARL and baselines.}
\end{figure}
```

### If you want a paper-like “heatmap” figure
The heatmap is generated only when you load **multiple strategies** (e.g., FIXED\_TIME + MARL\_DQN + MAX\_PRESSURE). It visualizes normalized mean metrics per strategy.


In [ ]:
# Optional (advanced): time-series plots
#
# Papers sometimes show dynamics over time (queues building/dissipating). Your repo can export CSVs
# for a single run (vehicle_data.csv / traffic_light_data.csv / detector_data.csv). If you want,
# we can add a dedicated "time-series" section later.
#
# For the report-quality baseline comparison, the episode-level plots + heatmap are typically enough.
pass


Using vehicle CSV: D:\Final Year Project\traffic-signal-control\output\vehicle_data.csv


,time,avg_waiting_time_t,p95_waiting_time_t,avg_speed_t,queued_vehicle_count_t,sum_co2_emission_t,sum_fuel_consumption_t,sum_nox_emission_t,sum_pmx_emission_t
0,5.0,0.400000,1.6,10.103090,0,0.0,0.0,0.0,0.0
1,10.0,1.250000,5.6,5.514559,1,0.0,0.0,0.0,0.0
2,15.0,1.727273,8.5,7.215333,1,0.0,0.0,0.0,0.0
3,20.0,1.785714,10.5,6.164400,2,0.0,0.0,0.0,0.0
4,25.0,2.529412,14.0,5.777929,3,0.0,0.0,0.0,0.0


In [ ]:
# Deprecated section (time-series plots).
# We intentionally keep the report notebook focused on *episode-level* plots + heatmap,
# because those align with the figures/tables most papers show for final comparisons.
pass


## Notes on paper-style comparison

To match the reporting style in your reference papers:
- Your **main Results** should be based on the **episode-level metric distribution** (mean  std, 95% CI) across many seeded runs.
- Use plots that complement the table:
  - ATT/AIP/ASIP vs episode (stability/robustness)
  - distribution plot or heatmap (multi-metric comparison)
  - emissions plots (CO$_2$/fuel) when enabled

The cells above already compute these and save figures for Overleaf.
is better”: show **A (final metrics)**.
- If you want to explain “why it’s better”: add **B (time-series)**.
- If reviewers question RL stability: add **C (episode curves)**.


In [ ]:
# Deprecated: the old plotly-based comparison helpers are removed.
# Use: plot_att_aip_asip_vs_episode / plot_fundamental_metrics_vs_episode / plot_strategy_heatmap.
pass
.Scatter(
            x=df["strategy"],
            y=df["mean"],
            mode="lines+markers",
            name=metric,
            error_y=dict(type="data", array=df["std"], visible=True),
        )
    )

    fig.update_layout(
        title=title or f"{metric}: mean ± std across strategies",
        xaxis_title="Strategy",
        yaxis_title=f"{metric} (mean ± std)",
    )
    fig.show()


# Example template: plug in your own results.
#
# For baselines, you typically create results by running each strategy multiple times
# (or using the Flask 'Academic Experiments' page), then saving per-run metrics.
#
# Here is a minimal placeholder structure:
strategy_to_runs_example = {
    # "FIXED_TIME": [ {"avg_waiting_time": ...}, {"avg_waiting_time": ...}, ... ],
    # "MAX_PRESSURE": [ ... ],
    # "DQN": [ ... ],
}

strategy_to_runs_example


SyntaxError: invalid syntax (1280939436.py, line 4)

In [ ]:
# Deprecated: old filename parsing logic.
# We now explicitly select eval JSON files in Cell 3 and infer labels via the JSON payload.
pass


## (Optional) Training curves

If you want a training-convergence figure for your dissertation (reward/loss/epsilon vs training episodes), we can add a dedicated training-curve notebook section later.

For most research-style reports, the priority is:
- final baseline comparison table (mean  std, 95% CI)
- stability/robustness plots (ATT/AIP/ASIP vs episode)
- environmental metrics (CO$_2$/fuel) if claimed


In [ ]:
# Deprecated section (training curves). Kept empty to avoid confusing plots.
pass


In [ ]:
# Deprecated section (training curves). Kept empty.
pass
ong)
    losses = th.get("losses") or []
    loss_df = pd.DataFrame({
        "train_step": np.arange(1, len(losses) + 1, dtype=int),
        "loss": pd.Series(losses, dtype=float) if losses else np.nan,
    })

    return ep_df, loss_df


def plot_training_curves(ckpt_path: Path, loss_smooth: int = 500):
    require_plotly()
    if torch is None:
        raise RuntimeError("torch not available in this kernel")

    info = load_training_history_from_checkpoint(ckpt_path)
    if not info:
        return

    ep_df, loss_df = training_history_to_dfs(info)
    title_prefix = ckpt_path.name

    # Reward vs episode
    if "episode_reward" in ep_df.columns and ep_df["episode_reward"].notna().any():
        tmp = ep_df[["episode", "episode_reward"]].copy()
        tmp["reward_smooth"] = tmp["episode_reward"].rolling(10, min_periods=1).mean()
        fig = px.line(tmp, x="episode", y=["episode_reward", "reward_smooth"], title=f"{title_prefix}: reward vs episode")
        fig.update_layout(xaxis_title="Episode", yaxis_title="Episode reward")
        fig.show()

    # Epsilon vs episode
    if "epsilon" in ep_df.columns and ep_df["epsilon"].notna().any():
        fig = px.line(ep_df, x="episode", y="epsilon", title=f"{title_prefix}: epsilon vs episode")
        fig.update_layout(xaxis_title="Episode", yaxis_title="Epsilon")
        fig.show()

    # Loss vs training step (downsample + smooth)
    if "loss" in loss_df.columns and loss_df["loss"].notna().any() and len(loss_df) > 0:
        # Downsample for speed
        step = max(1, len(loss_df) // 5000)
        d = loss_df.iloc[::step].copy()
        d["loss_smooth"] = d["loss"].rolling(loss_smooth // step if loss_smooth else 1, min_periods=1).mean()
        fig = px.line(d, x="train_step", y=["loss", "loss_smooth"], title=f"{title_prefix}: loss vs training step")
        fig.update_layout(xaxis_title="Training step", yaxis_title="Loss")
        fig.show()


# Pick a checkpoint to visualize
if MODEL_FILES:
    CKPT = MODEL_FILES[0]
    print("Using checkpoint:", CKPT)
    if torch is not None:
        plot_training_curves(CKPT)


## (Optional) Time-series overlays

This notebook intentionally focuses on episode-level results for paper-style reporting.
If you later want time-series overlays (queue/wait/speed over time for a single run), we can add a small separate notebook that reads `vehicle_data.csv` per strategy.


In [ ]:
# Deprecated section.
pass
, ts_df in ts_by_label.items():
        if y not in ts_df.columns:
            continue
        d = ts_df[["time", y]].copy()
        if smooth_window and smooth_window > 1:
            d[y] = d[y].rolling(smooth_window, min_periods=1).mean()
        d["label"] = label
        rows.append(d)

    if not rows:
        print(f"No data found for '{y}'. Available columns per label:")
        for label, ts_df in ts_by_label.items():
            print("-", label, ":", sorted(ts_df.columns))
        return

    long = pd.concat(rows, ignore_index=True)
    fig = px.line(long, x="time", y=y, color="label", title=title or f"{y} vs time (overlay)")
    fig.update_layout(xaxis_title="Simulation time (s)", yaxis_title=y)
    fig.show()


# Auto-detect multiple vehicle CSVs and group them by strategy token
veh_files_all = list_files("*vehicle*_data*.csv")
groups = group_vehicle_csvs_by_strategy(veh_files_all)
print({k: [p.name for p in v[:3]] for k, v in groups.items()})

# Build one time-series per strategy using the newest file in that group
# (You can change to v[0], v[1] to select different runs.)
ts_by_strategy = {}
for strat, files in groups.items():
    if not files:
        continue
    df = load_vehicle_csv(files[0])
    ts_by_strategy[strat] = vehicle_timeseries(df, queue_wait_threshold_s=5.0)

list(ts_by_strategy.keys())


In [ ]:
# Deprecated section.
pass
